# 13 — Decision Theory
**References:** Berger (1985) *Statistical Decision Theory* Ch. 1–4 · Casella & Berger (2002) Ch. 7 · Lehmann & Casella (1998) Ch. 5

## Narrative thread
```
Loss functions -> Risk -> Bayes risk -> Admissibility -> Minimax -> James-Stein
```

## 1. Decision-theoretic framework

| Component | Symbol | Definition |
|---|---|---|
| Parameter space | $\Theta$ | Set of possible states of nature |
| Action space | $\mathcal{A}$ | Set of possible decisions |
| Loss function | $L(\theta, a)$ | Cost of taking action $a$ when truth is $\theta$ |
| Estimator/rule | $\delta(\mathbf{X})$ | Map from data to action |
| Risk function | $R(\theta, \delta)$ | $E_\theta[L(\theta, \delta(\mathbf{X}))]$ |

**Common loss functions:**

| Loss | Formula | Optimal estimator |
|---|---|---|
| Squared error | $(\theta - a)^2$ | Posterior mean (Bayes) / MLE (freq.) |
| Absolute error | $|\theta - a|$ | Posterior median |
| 0-1 loss | $\mathbf{1}(\theta \neq a)$ | Posterior mode (MAP) |
| LINEX | $e^{c(a-\theta)} - c(a-\theta) - 1$ | Asymmetric — penalizes over/under differently |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── Risk functions: MLE vs Bayes shrinkage for Normal mean ────────────────
# X ~ N(theta, 1), loss = squared error
# MLE: delta_MLE(x) = x,  Risk = 1 (constant)
# Bayes (Normal prior N(0, tau^2)): delta_Bayes(x) = (tau^2/(1+tau^2)) * x
#   = shrinks x toward 0
#   Risk = tau^2/(1+tau^2) * (1 + theta^2/(1+tau^2))

thetas = np.linspace(-5, 5, 300)
tau_sq = 2.0
shrink = tau_sq / (1 + tau_sq)

risk_mle   = np.ones_like(thetas)
risk_bayes = shrink + (1 - shrink)**2 * thetas**2  # bias^2 + variance

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(thetas, risk_mle,   color='#f72585', lw=2.5, label='MLE: $R(\\theta)=1$ (unbiased, const. risk)')
ax.plot(thetas, risk_bayes, color='#4361ee', lw=2.5, label=f'Bayes (shrinkage $c={shrink:.2f}$): lower near 0')
ax.fill_between(thetas, risk_bayes, risk_mle,
                where=risk_bayes < risk_mle, alpha=0.2, color='#4361ee', label='Bayes wins region')
ax.fill_between(thetas, risk_bayes, risk_mle,
                where=risk_bayes >= risk_mle, alpha=0.2, color='#f72585', label='MLE wins region')
ax.set_xlabel('$\\theta$'); ax.set_ylabel('Risk $R(\\theta, \\delta)$')
ax.set_title(f'Risk functions: MLE vs Bayes shrinkage\n'
              f'$\\tau^2={tau_sq}$: Bayes wins for $|\\theta|$ small, MLE for large')
ax.legend(fontsize=8)

# ── Bayes risk (prior-averaged risk) ─────────────────────────────────────
tau_sqs = np.linspace(0.1, 10, 200)
bayes_risk_values = [tau2 / (1 + tau2) for tau2 in tau_sqs]  # Bayes risk = tau^2/(1+tau^2)
mle_bayes_risk    = [1.0] * len(tau_sqs)  # MLE Bayes risk (prior-averaged) = 1

ax2 = axes[1]
ax2.plot(tau_sqs, bayes_risk_values, color='#4361ee', lw=2.5, label='Bayes estimator (Bayes risk)')
ax2.axhline(1.0, color='#f72585', lw=2, linestyle='--', label='MLE Bayes risk = 1')
ax2.set_xlabel('$\\tau^2$ (prior variance)')
ax2.set_ylabel('Bayes risk = $E_\\pi[R(\\theta,\\delta)]$')
ax2.set_title('Bayes risk vs prior variance $\\tau^2$\n'
               'Bayes estimator always beats MLE in prior-averaged sense')
ax2.set_ylim(0, 1.2)
ax2.legend()

plt.suptitle('Decision Theory: Loss, Risk, and Bayes Risk', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Admissibility and the minimax criterion

**Admissibility:** $\delta$ is **inadmissible** if there exists $\delta'$ such that:
$$R(\theta, \delta') \leq R(\theta, \delta) \quad \forall\,\theta \quad \text{with strict inequality for some } \theta$$

**Minimax:** $\delta^*$ is **minimax** if it minimizes the worst-case risk:
$$\delta^* = \arg\min_\delta \sup_\theta R(\theta, \delta)$$

**Key results:**
- Every Bayes estimator with proper prior is **admissible**
- Least favorable prior: the prior $\pi^*$ that maximizes the Bayes risk → Bayes rule under $\pi^*$ is minimax
- **MLE for Normal mean (p=1):** $\delta(x) = x$ is both admissible and minimax

## 3. Stein's phenomenon and the James-Stein estimator

**Stein (1956) shocking result:** For estimating $\boldsymbol{\mu} \in \mathbb{R}^p$ with $p \geq 3$ under squared error loss, the MLE $\hat{\boldsymbol{\mu}} = \mathbf{X}$ is **inadmissible**!

**James-Stein estimator:**
$$\hat{\boldsymbol{\mu}}^{\text{JS}} = \left(1 - \frac{p-2}{\|\mathbf{X}\|^2}\right)\mathbf{X}$$

This dominates the MLE for all $\boldsymbol{\mu}$ when $p \geq 3$:
$$R(\boldsymbol{\mu}, \hat{\boldsymbol{\mu}}^{\text{JS}}) < R(\boldsymbol{\mu}, \mathbf{X}) = p \quad \forall\,\boldsymbol{\mu}$$

The saving is $p - R(\boldsymbol{\mu}, \hat{\boldsymbol{\mu}}^{\text{JS}}) = (p-2)^2 E[1/\|\mathbf{X}\|^2]$.

In [ ]:
# ── James-Stein estimator: risk reduction vs MLE ─────────────────────────
n_reps  = 50_000
ps_demo = [1, 2, 3, 5, 10, 20]
mu_norm_sq_vals = [0.0, 1.0, 4.0, 16.0]  # ||mu||^2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Risk vs ||mu||^2 for p=5
p_fixed   = 5
norm_sqs  = np.linspace(0, 30, 100)

risk_mle_th  = p_fixed * np.ones_like(norm_sqs)
# James-Stein risk (approximate formula)
risk_js_emp  = []
for ns in norm_sqs:
    # mu with ||mu||^2 = ns: put it all in first component
    mu = np.zeros(p_fixed); mu[0] = np.sqrt(ns)
    X  = mu + np.random.normal(0, 1, (n_reps, p_fixed))
    Xnorm2 = (X**2).sum(axis=1, keepdims=True)
    JS = (1 - (p_fixed - 2) / Xnorm2) * X
    risk_js_emp.append(((JS - mu)**2).sum(axis=1).mean())

axes[0].plot(norm_sqs, risk_mle_th, color='#f72585', lw=2.5, label=f'MLE risk = p = {p_fixed}')
axes[0].plot(norm_sqs, risk_js_emp, color='#4361ee', lw=2.5, label='James-Stein risk (simulated)')
axes[0].set_xlabel('$\\|\\boldsymbol{\\mu}\\|^2$'); axes[0].set_ylabel('Risk')
axes[0].set_title(f'James-Stein vs MLE — p={p_fixed}\n'
                   'JS uniformly dominates MLE for p≥3 (Stein phenomenon)')
axes[0].legend()

# Risk reduction vs dimension p
mu_fixed_norm2 = 0.0  # at mu=0, JS most beneficial
risk_reduction = []

for p_ in ps_demo:
    mu   = np.zeros(p_)
    X    = mu + np.random.normal(0, 1, (n_reps, p_))
    Xn2  = (X**2).sum(axis=1, keepdims=True)
    if p_ >= 3:
        JS   = (1 - (p_ - 2) / Xn2) * X
        r_js = ((JS - mu)**2).sum(axis=1).mean()
    else:
        r_js = p_  # MLE is admissible for p<=2
    risk_reduction.append((p_ - r_js) / p_ * 100)

bars = axes[1].bar([str(p) for p in ps_demo], risk_reduction,
                   color=['#ccc' if p<3 else '#4361ee' for p in ps_demo], alpha=0.8)
axes[1].axhline(0, color='k', lw=1)
for b, r, p_ in zip(bars, risk_reduction, ps_demo):
    axes[1].text(b.get_x()+b.get_width()/2, r+0.3, f'{r:.1f}%', ha='center', fontsize=9)
axes[1].set_xlabel('Dimension p'); axes[1].set_ylabel('Risk reduction vs MLE (%)')
axes[1].set_title('James-Stein risk reduction at μ=0\n'
                   'Gray: p<3 (JS not applicable); Blue: p≥3')

plt.suptitle("Stein's Phenomenon: MLE is inadmissible in ≥3 dimensions", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Connections to regularization

The James-Stein estimator is a form of **shrinkage** — it pulls estimates toward a target (here, zero). This connects to:

- **Ridge regression:** $\hat{\boldsymbol{\beta}}_{\text{ridge}} = (\mathbf{X}^\top\mathbf{X} + \lambda I)^{-1}\mathbf{X}^\top\mathbf{y}$ — shrinks toward zero under squared loss
- **Lasso:** $\ell_1$ regularization — sparsity-inducing shrinkage
- **Empirical Bayes:** Estimate the prior from data, then apply Bayes shrinkage

The decision-theoretic view unifies estimation and regularization under a common framework.

## 5. Key takeaways

| Concept | Statement |
|---|---|
| Risk | Expected loss $R(\theta,\delta) = E_\theta[L(\theta,\delta(X))]$ |
| Admissibility | No uniformly better estimator exists |
| Minimax | Minimizes worst-case risk |
| Bayes $\Rightarrow$ admissible | Proper priors always give admissible rules |
| Stein phenomenon | MLE inadmissible in $\geq 3$ dimensions |
| James-Stein | Shrinkage dominates MLE — foundation of regularization |

> **End of course.** You have covered: probability foundations, distribution theory, convergence, point estimation (UMVUE, Cramér-Rao), MLE theory, interval estimation, NP lemma, likelihood ratio tests, asymptotic theory, Bayesian inference, nonparametric methods, and decision theory.